In [0]:
# Imports 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SilverToGoldDelta").getOrCreate()

In [0]:
# Paths

BASE_PATH = "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net"

BRONZE_RAW_DELTA_PATH = f"{BASE_PATH}/bronze_delta/air_quality_raw/"
BRONZE_STRUCTURED_DELTA_PATH = f"{BASE_PATH}/bronze_delta/air_quality_structured/"
SILVER_DELTA_PATH = f"{BASE_PATH}/silver_delta/air_quality/"
GOLD_DELTA_BASE_PATH = f"{BASE_PATH}/gold_delta/"
CHECKPOINT_DELTA_BASE_PATH = f"{BASE_PATH}/checkpoints_delta/"

In [0]:
# Read silver Data

cleaned_df = spark.read.format("delta").load(SILVER_DELTA_PATH)

In [0]:
# Date extraction

cleaned_df = cleaned_df.withColumn(
    "measurement_date",
    F.col("measurement_timestamp").cast("date")
)

In [0]:
# Daily city aqi

daily_city_aqi_df = cleaned_df.groupBy(
    "city",
    "measurement_date"
).agg(
    F.round(F.avg("european_aqi"), 2).alias("avg_european_aqi"),
    F.max("european_aqi").alias("max_european_aqi"),
    F.round(F.avg("pm10"), 2).alias("avg_pm10"),
    F.max("pm10").alias("max_pm10"),
    F.round(F.avg("pm2_5")).alias("avg_pm2_5"),
    F.max("pm2_5").alias("max_pm2_5"),
    F.round(F.avg("nitrogen_dioxide")).alias("avg_nitrogen_dioxide"),
    F.max("nitrogen_dioxide").alias("max_nitrogen_dioxide"),
    F.count("*").alias("record_count")
).orderBy(
    "city",
    "measurement_date"
)

daily_city_aqi_df.show(truncate=False)

+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|city        |measurement_date|avg_european_aqi|max_european_aqi|avg_pm10|max_pm10|avg_pm2_5|max_pm2_5|avg_nitrogen_dioxide|max_nitrogen_dioxide|record_count|
+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|Athens      |2026-07-09      |39.36           |60              |17.19   |25.2    |12.0     |17.5     |36.0                |71.2                |22          |
|Berlin      |2026-07-09      |23.23           |30              |8.05    |11.0    |4.0      |4.1      |4.0                 |6.6                 |22          |
|London      |2026-07-09      |35.95           |63              |14.68   |16.9    |10.0     |10.6     |21.0                |39.9                |22          |
|Paris       |2026-07-09      |29.0           

In [0]:
# Daily city ranking by aqi

daily_city_aqi_rank_df = daily_city_aqi_df.withColumn(
    "rank",
    F.rank().over(Window.partitionBy(
        "city"
    ).orderBy(
        F.col("avg_european_aqi").desc()
    )
))

daily_city_aqi_rank_df.show(truncate=False)

+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+----+
|city        |measurement_date|avg_european_aqi|max_european_aqi|avg_pm10|max_pm10|avg_pm2_5|max_pm2_5|avg_nitrogen_dioxide|max_nitrogen_dioxide|record_count|rank|
+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+----+
|Athens      |2026-07-09      |39.36           |60              |17.19   |25.2    |12.0     |17.5     |36.0                |71.2                |22          |1   |
|Berlin      |2026-07-09      |23.23           |30              |8.05    |11.0    |4.0      |4.1      |4.0                 |6.6                 |22          |1   |
|London      |2026-07-09      |35.95           |63              |14.68   |16.9    |10.0     |10.6     |21.0                |39.9                |22          |1   |
|Paris       |20

In [0]:
# Data freshness

data_freshness_df = cleaned_df.groupBy(
    "city"
).agg(
    F.max("ingestion_timestamp_utc").alias("latest_ingestion_timestamp_utc"),
    F.max("measurement_timestamp").alias("latest_measurement_timestamp")
    )

data_freshness_df.show(truncate=False)

+------------+------------------------------+----------------------------+
|city        |latest_ingestion_timestamp_utc|latest_measurement_timestamp|
+------------+------------------------------+----------------------------+
|Thessaloniki|2026-07-09 20:58:22.148159    |2026-07-09 21:00:00         |
|London      |2026-07-09 20:58:22.555047    |2026-07-09 21:00:00         |
|Paris       |2026-07-09 20:58:23.066659    |2026-07-09 21:00:00         |
|Athens      |2026-07-09 20:58:21.635676    |2026-07-09 21:00:00         |
|Berlin      |2026-07-09 20:58:23.578567    |2026-07-09 21:00:00         |
+------------+------------------------------+----------------------------+



In [0]:
# Data completeness

data_completeness_df = cleaned_df.groupBy(
        "city",
        "measurement_date"
    ).agg(
        F.count("*").alias("actual_records")
    ).withColumn("expected_records", F.lit(24)).withColumn(
        "completeness_pct", F.round(F.col("actual_records") / F.col("expected_records") * 100, 2))
    
data_completeness_df.show(truncate=False)

+------------+----------------+--------------+----------------+----------------+
|city        |measurement_date|actual_records|expected_records|completeness_pct|
+------------+----------------+--------------+----------------+----------------+
|Athens      |2026-07-09      |22            |24              |91.67           |
|Thessaloniki|2026-07-09      |22            |24              |91.67           |
|Paris       |2026-07-09      |22            |24              |91.67           |
|London      |2026-07-09      |22            |24              |91.67           |
|Berlin      |2026-07-09      |22            |24              |91.67           |
+------------+----------------+--------------+----------------+----------------+



In [0]:
# Writing to Gold Delta

daily_city_aqi_df.write.format("delta").mode("overwrite").save(
    GOLD_DELTA_BASE_PATH + "daily_city_aqi/"
)

daily_city_aqi_rank_df.write.format("delta").mode("overwrite").save(
    GOLD_DELTA_BASE_PATH + "daily_city_ranking/"
)

data_freshness_df.write.format("delta").mode("overwrite").save(
    GOLD_DELTA_BASE_PATH + "data_freshness/"
)

data_completeness_df.write.format("delta").mode("overwrite").save(
    GOLD_DELTA_BASE_PATH + "data_completeness/"
)

In [0]:
# Test

spark.read.format("delta").load(
    GOLD_DELTA_BASE_PATH + "daily_city_aqi/"
).show(truncate=False)

+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|city        |measurement_date|avg_european_aqi|max_european_aqi|avg_pm10|max_pm10|avg_pm2_5|max_pm2_5|avg_nitrogen_dioxide|max_nitrogen_dioxide|record_count|
+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|Athens      |2026-07-09      |39.36           |60              |17.19   |25.2    |12.0     |17.5     |36.0                |71.2                |22          |
|Berlin      |2026-07-09      |23.23           |30              |8.05    |11.0    |4.0      |4.1      |4.0                 |6.6                 |22          |
|London      |2026-07-09      |35.95           |63              |14.68   |16.9    |10.0     |10.6     |21.0                |39.9                |22          |
|Paris       |2026-07-09      |29.0           